# 09 · Grouped out-of-fold predictions and matched support selectors

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

Use the flagship benchmark for claim support. Foundation annotation agreement is a different target, analysed separately in notebook 21.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Configure the prespecified run and context

In [ ]:
from oncoplate.inputs import load_context
from oncoplate.pipeline import spec_from_cfg
from oncoplate.governance import require_gate
require_gate(cfg,'domain_schema')
assert cfg['study']['dataset']!='foodnextdb','FoodNExTDB lacks independent claim-support references. Use notebook 21 for foundation agreement.'
records,targets,states,rules=load_context(cfg,require_review=True)
RUN_SEED=0  # Repeat 0–4; do not select the best seed.
spec=spec_from_cfg(cfg,'dinov2_vits14','finetune','joint',RUN_SEED)
context=read_json(p['private']/"protocol_context.json")

## 2. Generate fold-local focal claims
Every held group is excluded from backbone fitting AND inner early stopping. The fold-specific runs resume after interruption.

In [ ]:
from oncoplate.evaluation import oof_claim_candidates
from oncoplate.benchmark import make_rating_jobs
oof=oof_claim_candidates(cfg,spec,states,rules,asof=context['asof'],input_mode=context['input_mode'])
root=p['runs']/'oof'/spec.run_id
jobs=make_rating_jobs(oof,root/'oof_claim_rating_jobs.csv')
print('Independent rating jobs:',root/'oof_claim_rating_jobs.csv',len(jobs))

## 3. Import independently adjudicated OOF ratings
Complete this outside the model output interface, retain both reference axes, then rerun this cell. Ratings are not generated from PCSI eligibility.

In [ ]:
from oncoplate.benchmark import adjudicated_ratings
from oncoplate.selection import train_selectors
ratings_path=root/'oof_claim_ratings_adjudicated.csv'
assert ratings_path.exists(),'Independent OOF claim ratings are required before training support selectors.'
ratings=adjudicated_ratings(read_table(ratings_path))
selector_dir=p['private']/'selectors'/spec.run_id
train_selectors(oof,ratings,selector_dir)
print('Saved matched selectors:',selector_dir)

## 4. Optional neural-selector capacity ablation
Same feature contracts; fixed architecture/epochs. Keep the logistic comparison as the registered principal implementation unless amended before testing.

In [ ]:
from oncoplate.selection import SoftMLP,GENERIC_FEATURES,PROVENANCE_FEATURES
from oncoplate.benchmark import join_ratings
joined=join_ratings(oof,ratings);joined=joined[joined.support_status.ne('noninformative')]
y=joined.support_status.eq('supported').to_numpy(float)
for name,features in [('generic',GENERIC_FEATURES),('pcsi',GENERIC_FEATURES+PROVENANCE_FEATURES)]:
    m=SoftMLP(seed=RUN_SEED).fit(joined[features].to_numpy(float),y)
    m.save(selector_dir/f'{name}_mlp.json',features,{'target':'independently_adjudicated_OOF','seed':RUN_SEED})

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
